# Hepatitis Dataset ML
Data Cleaning, Outlier Removal, Transformation and Model Comparison

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

## Load Dataset (No Header)

In [ ]:
df = pd.read_csv("../DSBDALExam DataSets/Hepatitis/hepatitis.csv", header=None)

df.columns = [
    'class','age','sex','steroid','antivirals','fatigue','malaise','anorexia',
    'liver_big','liver_firm','spleen_palpable','spiders','ascites','varices',
    'bilirubin','alk_phosphate','SGOT','albumin','protime','histology'
]


print(df['class'].value_counts())

class
2    123
1     32
Name: count, dtype: int64


## q. Data Cleaning

In [ ]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert numeric columns
num_cols = ['age','bilirubin','alk_phosphate','SGOT','albumin','protime']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values instead of dropping
df.fillna(df.mean(numeric_only=True), inplace=True)

# Remove negative values
df = df[(df[num_cols] >= 0).all(axis=1)]

df.shape

(155, 20)

## r. Outlier Removal

In [ ]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df[num_cols] < (Q1 - 1.5 * IQR)) | 
          (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

df.shape

(84, 20)

## s. Data Transformation

In [ ]:
# Convert target properly
df['class'] = df['class'].apply(lambda x: 1 if x == 2 else 0)

# Encoding
df = pd.get_dummies(df, drop_first=True)

# Split
X = df.drop('class', axis=1)
y = df['class']

print("Class distribution:", y.value_counts())

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

Class distribution: class
1    123
0     32
Name: count, dtype: int64


## t. Model Building and Comparison

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_acc = accuracy_score(y_test, nb.predict(X_test))

print("Logistic Regression Accuracy:", lr_acc)
print("Naive Bayes Accuracy:", nb_acc)

Logistic Regression Accuracy: 0.9032258064516129
Naive Bayes Accuracy: 0.7096774193548387
